In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
import xgboost as xgb
from xgboost import XGBRegressor

In [2]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelXG/ModelXG.json")

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[1.834995150566101, 1.6334861516952515, 2.6028685569763184, 2.4011988639831543, 2.8215129375457764, 3.3295278549194336, 2.2339842319488525, 3.3295278549194336, 2.7023568153381348, 3.0087499618530273, 4.517553806304932, 1.309707760810852, 1.7944164276123047, 2.3639042377471924, 2.0219292640686035, 2.9517130851745605, 2.961571216583252, 3.0087499618530273, 3.3295278549194336, 2.0219292640686035, 1.8924477100372314, 3.3295278549194336, 1.7914141416549683, 1.9461948871612549, 2.671940565109253, 1.8603394031524658, 2.0219292640686035, 2.3438973426818848, 2.8215129375457764, 4.402780055999756, 2.818509578704834, 2.145622730255127, 3.6452364921569824, 3.7358412742614746, 2.414212226867676, 2.8215129375457764, 2.818509578704834, 1.8633424043655396, 3.7408597469329834, 2.200052499771118, 4.517553806304932, 4.517553806304932, 3.6452364921569824, 1.6304852962493896, 4.279089450836182, 1.9491932392120361, 2.041816473007202, 4.206354141235352, 3.4282963275909424, 1.9344758987426758, 1.7735464572906

In [5]:
np.average(y_max_arr)

np.float32(2.87356)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelXG/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)